# Entrenamiento v2 — TF-IDF + Features manuales

En este notebook mejoramos el modelo baseline añadiendo features manuales al TF-IDF.

**Baseline (notebook 3):** LogisticRegression + TF-IDF → F1-macro validación: 0.87

**Mejoras en este notebook:**
1. Features manuales: longitud de texto + conteo de keywords de dominio por clase
2. Concatenación TF-IDF + features manuales
3. Re-entrenamiento de LogisticRegression
4. Comparativa con baseline
5. Registro en MLflow (Experimento 1)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os

# Localizar src/classifier/ de forma robusta y ajustar cwd al directorio
# de este notebook para que rutas relativas (datasets/, data/, model/) funcionen
# independientemente de desde donde se lance Jupyter/VS Code.
_cwd = os.getcwd()
_candidates = [
    os.path.join(_cwd, "src", "classifier"),
    os.path.abspath(".."),
    os.path.abspath("."),
]
for _p in _candidates:
    if os.path.isfile(os.path.join(_p, "functions.py")):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        # Cambiar cwd al directorio de este notebook
        os.chdir(os.path.join(_p, "classifier_ultimo_dataset"))
        break

import functions  # noqa: E402
functions.MLFLOW_EXPERIMENT = "clasificador_riesgo_ultimo_dataset"
functions._DATASET_TAGS = {"dataset_type": "ultimo", "dataset_source": "dataset_sintetico_v2"}

## 1. Carga de datos

In [3]:
import pandas as pd

train_df = pd.read_csv("data/processed/train.csv")
val_df = pd.read_csv("data/processed/validation.csv")
test_df = pd.read_csv("data/processed/test.csv")

X_train = train_df["text_final"]
y_train = train_df["etiqueta"]
X_val = val_df["text_final"]
y_val = val_df["etiqueta"]
X_test = test_df["text_final"]
y_test = test_df["etiqueta"]

print(f"Train: {len(X_train)} muestras")
print(f"Validation: {len(X_val)} muestras")
print(f"Test: {len(X_test)} muestras")

Train: 199 muestras
Validation: 43 muestras
Test: 43 muestras


## 2. Vectorización TF-IDF (misma config que baseline)

In [4]:
import joblib

# Cargar el TF-IDF ya ajustado (nb3 o nb7 lo entrenan y guardan con 5 000 términos).
# NO re-ajustamos aquí para que modelo_logreg_v2 sea compatible con el resto de
# notebooks que comparten el mismo tfidf_vectorizer.joblib.
tfidf = joblib.load("model/tfidf_vectorizer.joblib")
X_train_tfidf = tfidf.transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

print(f"Vocabulario TF-IDF: {len(tfidf.vocabulary_)} términos  (cargado desde model/)")
print(f"Forma train: {X_train_tfidf.shape}")
print(f"Forma validation: {X_val_tfidf.shape}")
print(f"Forma test: {X_test_tfidf.shape}")

Vocabulario TF-IDF: 1700 términos  (cargado desde model/)
Forma train: (199, 1700)
Forma validation: (43, 1700)
Forma test: (43, 1700)


## 3. Features manuales

In [5]:
from functions import crear_features_manuales, combinar_features, KEYWORDS_DOMINIO

# Generar features manuales para cada conjunto
feat_train = crear_features_manuales(X_train)
feat_val = crear_features_manuales(X_val)
feat_test = crear_features_manuales(X_test)

print(f"Features manuales: {list(feat_train.columns)}")
print(f"Forma: {feat_train.shape}")
print()
print(feat_train.describe().round(2))

Features manuales: ['num_palabras', 'num_caracteres', 'kw_inaceptable', 'kw_alto_riesgo', 'kw_riesgo_limitado', 'kw_riesgo_minimo', 'kw_salvaguarda']
Forma: (199, 7)

       num_palabras  num_caracteres  kw_inaceptable  kw_alto_riesgo  \
count        199.00          199.00          199.00          199.00   
mean          11.22          104.36            0.03            0.09   
std            2.04           20.36            0.16            0.29   
min            7.00           63.00            0.00            0.00   
25%           10.00           89.00            0.00            0.00   
50%           11.00          104.00            0.00            0.00   
75%           12.00          117.00            0.00            0.00   
max           18.00          159.00            1.00            1.00   

       kw_riesgo_limitado  kw_riesgo_minimo  kw_salvaguarda  
count              199.00            199.00          199.00  
mean                 0.07              0.22            0.02  
std    

In [6]:
# Concatenar TF-IDF + features manuales
X_train_combined = combinar_features(X_train_tfidf, feat_train)
X_val_combined = combinar_features(X_val_tfidf, feat_val)
X_test_combined = combinar_features(X_test_tfidf, feat_test)

print(f"Forma train combinada: {X_train_combined.shape}")
print(f"Forma validation combinada: {X_val_combined.shape}")
print(f"Forma test combinada: {X_test_combined.shape}")
print(f"  (TF-IDF: {X_train_tfidf.shape[1]} + manuales: {feat_train.shape[1]} = {X_train_combined.shape[1]})")

Forma train combinada: (199, 1707)
Forma validation combinada: (43, 1707)
Forma test combinada: (43, 1707)
  (TF-IDF: 1700 + manuales: 7 = 1707)


## 4. Entrenamiento LogisticRegression con features combinadas

In [7]:
from functions import entrenar_modelo_baseline

modelo_v2 = entrenar_modelo_baseline(X_train_combined, y_train, X_val_combined, y_val)

=== Resultados en VALIDACIÓN ===

                 precision    recall  f1-score   support

    alto_riesgo       1.00      0.82      0.90        11
    inaceptable       0.91      1.00      0.95        10
riesgo_limitado       0.85      1.00      0.92        11
  riesgo_minimo       1.00      0.91      0.95        11

       accuracy                           0.93        43
      macro avg       0.94      0.93      0.93        43
   weighted avg       0.94      0.93      0.93        43

F1-score macro (validación): 0.9304


## 5. Comparativa con baseline

In [8]:
from sklearn.metrics import f1_score

# Métricas del baseline (notebook 3)
BASELINE_F1_MACRO = 0.8698
BASELINE_ACCURACY = 0.87

# Métricas del modelo v2
y_val_pred_v2 = modelo_v2.predict(X_val_combined)
f1_v2 = f1_score(y_val, y_val_pred_v2, average="macro")
acc_v2 = (y_val_pred_v2 == y_val).mean()

print("=== COMPARATIVA VALIDACIÓN ===")
print(f"{'Métrica':<20} {'Baseline':>10} {'V2 (+ features)':>16} {'Diferencia':>12}")
print("-" * 60)
print(f"{'F1-macro':<20} {BASELINE_F1_MACRO:>10.4f} {f1_v2:>16.4f} {f1_v2 - BASELINE_F1_MACRO:>+12.4f}")
print(f"{'Accuracy':<20} {BASELINE_ACCURACY:>10.4f} {acc_v2:>16.4f} {acc_v2 - BASELINE_ACCURACY:>+12.4f}")

=== COMPARATIVA VALIDACIÓN ===
Métrica                Baseline  V2 (+ features)   Diferencia
------------------------------------------------------------
F1-macro                 0.8698           0.9304      +0.0606
Accuracy                 0.8700           0.9302      +0.0602


## 6. Guardar artefactos

In [9]:
import joblib
import os

output_dir = "model"
os.makedirs(output_dir, exist_ok=True)

joblib.dump(modelo_v2, os.path.join(output_dir, "modelo_logreg_v2.joblib"))
joblib.dump(modelo_v2, os.path.join(output_dir, "modelo_clasificador.joblib"))
# tfidf_vectorizer.joblib NO se sobreescribe aquí — es propiedad de nb3/nb7

print("Modelo v2 guardado en: model/modelo_logreg_v2.joblib")
print("Modelo v2 guardado en: model/modelo_clasificador.joblib")

Modelo v2 guardado en: model/modelo_logreg_v2.joblib
Modelo v2 guardado en: model/modelo_clasificador.joblib


## 7. Registro en MLflow — Experimento 1

In [10]:
# ── MLflow (solo falla esta celda si el servidor no está disponible) ──
import mlflow
from functions import configure_mlflow, MLFLOW_EXPERIMENT

try:
    configure_mlflow()
    mlflow.set_experiment(MLFLOW_EXPERIMENT)

    with mlflow.start_run(run_name="v2_tfidf_features_manuales"):
        mlflow.log_param("tfidf_max_features",     5000)
        mlflow.log_param("tfidf_ngram_range",      "(1, 2)")
        mlflow.log_param("tfidf_sublinear_tf",     True)
        mlflow.log_param("modelo",                 "LogisticRegression")
        mlflow.log_param("max_iter",               1000)
        mlflow.log_param("features_manuales",      list(feat_train.columns))
        mlflow.log_param("n_keywords_por_clase",   len(list(KEYWORDS_DOMINIO.values())[0]))
        mlflow.log_param("total_features",         X_train_combined.shape[1])

        mlflow.log_metric("val_f1_macro",          f1_v2)
        mlflow.log_metric("val_accuracy",          acc_v2)
        mlflow.log_metric("baseline_f1_macro",     BASELINE_F1_MACRO)

        print("✓ Exp 1 registrado en MLflow")
        print(f"  Val F1-macro: {f1_v2:.4f}")
        print(f"  Run ID: {mlflow.active_run().info.run_id}")
except Exception as e:
    print(f"⚠ MLflow no disponible: {e}")

Password obtenida desde variable de entorno local.
MLflow configurado correctamente → https://18.201.64.41/
⚠ MLflow no disponible: API request to https://18.201.64.41/api/2.0/mlflow/experiments/get-by-name failed with timeout exception HTTPSConnectionPool(host='18.201.64.41', port=443): Max retries exceeded with url: /api/2.0/mlflow/experiments/get-by-name?experiment_name=clasificador_riesgo_ultimo_dataset (Caused by ConnectTimeoutError(<HTTPSConnection(host='18.201.64.41', port=443) at 0x1214ef86e40>, 'Connection to 18.201.64.41 timed out. (connect timeout=120)')). To increase the timeout, set the environment variable MLFLOW_HTTP_REQUEST_TIMEOUT (default: 120, type: int) to a larger value.
